## Revisions

Revision: Changed img embedding concatination from seq_len(dim=1 at t=0) to embedding_dim(dim=2)

In [ ]:
from IPython.display import clear_output, display

In [ ]:
# %pip install torch torchvision pillow spacy numpy
# %pip install torchtext
# %pip install pycocotools

In [ ]:
import os
import math
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import CocoCaptions

from tqdm import tqdm

from PIL import Image
import spacy

In [ ]:
dataset_variant = 'val2017'

## Downloading the data

In [ ]:
# Define paths for dataset and annotations
data_dir = './data'
images_dir = os.path.join(data_dir, dataset_variant)
annotations_dir = os.path.join(data_dir, 'annotations')

# Create directories if they don't exist
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
if not os.path.exists(images_dir):
    os.makedirs(images_dir)
if not os.path.exists(annotations_dir):
    os.makedirs(annotations_dir)

# Download dataset
!wget http://images.cocodataset.org/zips/{dataset_variant}.zip -P {data_dir}

# Unzip dataset
!unzip {data_dir}/{dataset_variant}.zip -d {data_dir}

clear_output()


In [ ]:
# Download annotations
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P {annotations_dir}

# # Unzip annotations
!unzip {annotations_dir}/annotations_trainval2017.zip -d {annotations_dir}

clear_output()

## Loading the Dataset

In [ ]:
transform = transforms.Compose(
        [
            transforms.Resize((299, 299)),
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ]
    )

# Load MS-COCO dataset
train_dataset = CocoCaptions(root=f'./data/{dataset_variant}', annFile=f'./data/annotations/annotations/captions_{dataset_variant}.json', transform=transform)

## Building the tokenizer and vocabulary

In [ ]:
spacy_eng = spacy.load("en_core_web_sm")

In [ ]:
def word_tokenize(text):
    return [tok.text.lower() for tok in spacy_eng.tokenizer(text)]

In [ ]:
# Define the vocabulary and tokenizer
word_to_index = {'<PAD>':0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
index_to_word = {it: k for k, it in word_to_index.items()}
word_freq = {}
caption_lengths = []


# Tokenize captions and build vocabulary
for _, captions in tqdm(train_dataset):
    for caption in captions:
        caption = f'{caption}'
        caption_lengths.append(len(caption))
        tokens = word_tokenize(caption.lower())
        for token in tokens:
            if token not in word_to_index:
                idx = len(word_to_index)
                word_to_index[token] = idx
                index_to_word[idx] = token
                word_freq[token] = 1
            else:
                word_freq[token] += 1

In [ ]:
word_tokenize('<SOS> hi, my friend <EOS>')  # We will manually add tokens for <EOS> and <SOS> etc after tokenization to avoid them breaking up.

## Defining the Model

In [ ]:
class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()

        self.resnet = models.resnet50(pretrained=True).requires_grad_(False)  # resnet embedding backbone
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, embed_size)
        self.relu = nn.ReLU()
        self.times = []
        self.dropout = nn.Dropout(0.5)

    def forward(self, images):

        features = self.resnet(images)
        return self.dropout(self.relu(features))


class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(DecoderRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)

        self.rnn = nn.LSTM(2*embed_size, hidden_size, num_layers, batch_first=True)  # 2* because we'll concat image embeddings with every token's embedding

        # NOTE: We can replace the LSTM architecture by RNN or GRU by replacing the above line by one of the following

        # self.rnn = nn.RNN(embed_size, hidden_size, num_layers, batch_first=True)
        # self.rnn = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)

        self.linear = nn.Sequential(
            nn.Linear(hidden_size, 1024),
            nn.Linear(1024, vocab_size)
        )
        self.dropout = nn.Dropout(0.5)

    def forward(self, features, captions):

        embeddings = self.dropout(self.embed(captions))
        seq_length = embeddings.shape[-2]

        repeat_shape = [1]*embeddings.ndim
        repeat_shape[-2] = seq_length
        features = features.unsqueeze(-2).repeat(repeat_shape)  # add sequence length dimension. Make it the same shape as caption embeddings.

        combined_embeddings = torch.cat((embeddings, features), dim=-1)

        outputs, _ = self.rnn(combined_embeddings)
        outputs = self.linear(outputs)
        return outputs


class ImageCaptioner(nn.Module):

    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(ImageCaptioner, self).__init__()
        self.encoder = EncoderCNN(embed_size)
        self.decoder = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers)

    def forward(self, images, captions):
        features = self.encoder(images)
        outputs = self.decoder(features, captions)
        return outputs

    def caption_image(self, image, max_length=50):

        result_caption = []

        with torch.no_grad():

            img_embedding = self.encoder(image)
            x = self.decoder.embed(torch.tensor([word_to_index['<SOS>']]).to(img_embedding.device))

            states = None

            for _ in range(max_length):

                x = torch.cat((x, img_embedding), dim=-1)

                output, states = self.decoder.rnn(x, states)
                output = self.decoder.linear(output.squeeze(0))

                predicted = output.argmax(0)
                pred_token = index_to_word[predicted.item()]
                result_caption.append(pred_token)
                x = self.decoder.embed(predicted).unsqueeze(0)

                if pred_token == "<EOS>":
                    break

        return result_caption

## Defining the dataset

In [ ]:
def convert_sentence_to_idxs(sentence):

    words = word_tokenize(sentence)
    idxs = [word_to_index[word] for word in words]

    return idxs


def convert_idxs_to_sentence(idxs):

    words = [index_to_word[idx] for idx in idxs]
    return ' '.join(words)


class CoCoCaptionsDataset(Dataset):

    def __init__(self, default_coco_dataset, max_seq_len):

        self.coco = default_coco_dataset
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.coco)

    def __getitem__(self, idx):

        img, captions = self.coco[idx]

        # caption = random.choice(captions, k=1)[0]
        caption = captions[0]
        caption_idxs = convert_sentence_to_idxs(caption.lower().strip())
        if len(caption_idxs) > self.max_seq_len - 2:  # 2 for SOS and EOS
            caption_idxs = caption_idxs[:self.max_seq_len-2]

        padding_len = self.max_seq_len - len(caption_idxs) - 2  # need to pad to make it to seq_len. All captions Need to be of same length so they can be stacked

        caption = (
            [word_to_index['<SOS>']]+
            caption_idxs+
            [word_to_index['<EOS>']]+
            [word_to_index['<PAD>']]*padding_len
        )

        caption = torch.Tensor(caption)

        return img, caption


batch_size = 128
fixed_train_dataset = CoCoCaptionsDataset(train_dataset, max_seq_len=50)
train_loader = DataLoader(fixed_train_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
embed_size = 512
hidden_size = 256
vocab_size = len(word_to_index)
num_decoder_layers = 1
learning_rate = 1e-4
num_epochs = 100

In [ ]:
# initialize model, loss etc
model = ImageCaptioner(embed_size, hidden_size, vocab_size, num_decoder_layers).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=word_to_index['<PAD>'])  # ignore pad token loss calculations
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Only finetune the CNN
for name, param in model.encoder.resnet.named_parameters():
    if "fc.weight" in name or "fc.bias" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

## Pre-training Testing

In [ ]:
test_img_paths = ['data/val2017/000000000139.jpg', 'data/val2017/000000000632.jpg', 'data/val2017/000000000724.jpg']
imgs_pil = [Image.open(path).convert('RGB') for path in test_img_paths]
imgs_test = [transform(im_pil).to(device) for im_pil in imgs_pil]

In [ ]:
imgs = torch.stack(imgs_test, 0)
model.eval()
captions = []

for img in imgs_test:
    with torch.no_grad():
        caption = model.caption_image(img.unsqueeze(0))
        caption = ' '.join(caption)
        captions.append(caption)

print(('\n'+'-'*20+'\n').join(captions))

## Training the model

In [ ]:
for epoch in range(num_epochs):

    model.train()

    for idx, (imgs, captions) in tqdm(
        enumerate(train_loader), total=len(train_loader), leave=False
    ):
        imgs = imgs.to(device)
        captions = captions.to(device).type(torch.long)

        outputs = model(imgs, captions[:, :-1])
        loss = criterion(
            outputs.reshape(-1, outputs.shape[2]), captions[..., 1:].reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'\nEpoch: {epoch+1}/{num_epochs}', "Training loss: ", loss.item())

In [ ]:
!ls data/val2017/ | head -20

In [ ]:
!ls data/

In [ ]:
img = Image.open('data/val2017/000000001490.jpg').convert('RGB')
img

In [ ]:
img_t = transform(img).to(device).unsqueeze(0)
model.eval()
with torch.no_grad():
    caption = model.caption_image(img_t)

caption

In [ ]:
for img, img_pil in zip(imgs_test, imgs_pil):
    with torch.no_grad():
        caption = model.caption_image(img.unsqueeze(0))
        caption = ' '.join(caption)

        display(img_pil)
        print(caption)
        print('-'*20)